# 09 · Overlay, late fusion over the frozen M1

Runs the late-fusion grid, four pure functions of the frozen M1 predictions and the chains' pooled evidence probability q, the chain's own predicted observable read as an error probability, P(present) as P(incorrect). Nothing refits, M1 trains once and its predictions freeze, the chains fit once under the chain of record and freeze, fitted fusion scalars estimate on the training predictions by Bernoulli log-likelihood, so attribution is algebraic, no joint EM, no compensation, no initialization sensitivity, the delta against the baseline row is the fusion's alone, and the baseline row reproduces M1's paper-aligned trio exactly, 60.65 accuracy, 64.28 AUC, 55.60 f1.

**The methods**, each crossed with every pooling:

- pseudo_kc, the chain as one more KC in the turn average, (K p + (1 minus q)) over (K + 1), no parameters.
- linear_pool, (1 minus w) p + w (1 minus q), w fitted.
- product, p (1 minus q), the conjunctive overlay, no parameters.
- frozen_gate, p (1 minus beta q), beta fitted, the emission functional form on frozen predictions, its beta against the early-fusion 0.178 measures what the joint refit absorbed.

**Reading order.** The diagnostic in section 2 first, the residual correlation computed on our own artifacts, it is the information ceiling every row below plays under. Then the baseline row, then the grid, then the deltas. The standing caveats carry, selection on the test set makes deltas descriptive, and effect sizes at stake sit at the order of seed and optimizer variation.

## 1. Setup

M1 trained once, chains fitted once, both frozen into the overlay.

In [1]:
import numpy as np
import pandas as pd
from scripts.load_data import load_paper_filtered_data
from scripts.bkt_model import BKTModel
from scripts.chain import TriggerChain
from scripts.misconception_chains import MisconceptionChains
from scripts.emission_integration_pooled import CHAIN_OF_RECORD
from scripts.overlay import Overlay, METHODS

train_df = load_paper_filtered_data("data/mathdial_train.csv")
test_df = load_paper_filtered_data("data/mathdial_test.csv")

m1 = BKTModel(train_df, test_df).train()
chains = MisconceptionChains(train_df, test_df, chain_class=TriggerChain,
                             chain_kwargs=dict(CHAIN_OF_RECORD))
chains.run()
overlay = Overlay(train_df, test_df, chains=chains, m1=m1)

## 2. The residual budget, computed on our own artifacts

The residual correlation between the pooled q and M1's own prediction error, on train and test. This number was previously adopted from external review at 0.06, here it is measured on the artifacts every row below uses. It is a warning-level budget for linear-ish fusion of these two frozen signals, not a mathematical bound on nonlinear methods, near zero means the grid below is an existence test expected to come back empty.

In [2]:
from scripts.pooling import Pooling

for split in ("train", "test"):
    frame = overlay.frames[split]
    q = overlay._pooled_q(frame, "max")
    error = frame["correct"].to_numpy() == 0
    residual = error.astype(float) - (1 - frame["prediction"].to_numpy())
    print(f"{split}: corr(q, error) "
          f"{np.corrcoef(q, error)[0, 1]:+.4f}, "
          f"corr(q, M1 residual) {np.corrcoef(q, residual)[0, 1]:+.4f}, "
          f"corr(q, M1 predicted error) "
          f"{np.corrcoef(q, 1 - frame['prediction'].to_numpy())[0, 1]:+.4f}")

train: corr(q, error) +0.1063, corr(q, M1 residual) +0.0502, corr(q, M1 predicted error) +0.2646
test: corr(q, error) +0.1105, corr(q, M1 residual) +0.0513, corr(q, M1 predicted error) +0.2962


## 3. The grid

Seventeen rows, the baseline plus four methods by four poolings, loop-built.

In [3]:
POOLINGS = ["max", "noisy_or", "top2_or", "mean"]

rows = [overlay.evaluate("baseline")]
for method in METHODS:
    for pooling in POOLINGS:
        rows.append(overlay.evaluate(method, pooling))
        print(f"{method:12s} {pooling:9s} {rows[-1]}")

results = pd.DataFrame(rows)
results["row"] = results["method"] + results["pooling"].map(
    lambda p: "" if pd.isna(p) else f", {p}")
results = results.set_index("row")
results[["method", "pooling", "parameter", "accuracy", "auc", "f1",
         "turns"]]

pseudo_kc    max       {'accuracy': 0.5804, 'auc': 0.6369, 'f1': 0.4003, 'turns': 1985, 'method': 'pseudo_kc', 'pooling': 'max', 'parameter': None}
pseudo_kc    noisy_or  {'accuracy': 0.5778, 'auc': 0.6383, 'f1': 0.3623, 'turns': 1985, 'method': 'pseudo_kc', 'pooling': 'noisy_or', 'parameter': None}
pseudo_kc    top2_or   {'accuracy': 0.5804, 'auc': 0.6381, 'f1': 0.3861, 'turns': 1985, 'method': 'pseudo_kc', 'pooling': 'top2_or', 'parameter': None}
pseudo_kc    mean      {'accuracy': 0.5254, 'auc': 0.6325, 'f1': 0.6366, 'turns': 1985, 'method': 'pseudo_kc', 'pooling': 'mean', 'parameter': None}
linear_pool  max       {'accuracy': 0.6081, 'auc': 0.6441, 'f1': 0.545, 'turns': 1985, 'method': 'linear_pool', 'pooling': 'max', 'parameter': 0.0303}
linear_pool  noisy_or  {'accuracy': 0.6055, 'auc': 0.6434, 'f1': 0.545, 'turns': 1985, 'method': 'linear_pool', 'pooling': 'noisy_or', 'parameter': 0.0181}
linear_pool  top2_or   {'accuracy': 0.6071, 'auc': 0.6439, 'f1': 0.5449, 'turns': 1985, 'me

,method,pooling,parameter,accuracy,auc,f1,turns
row,,,,,,,
baseline,baseline,NaN,NaN,0.6065,0.6428,0.5560,1985
"pseudo_kc, max",pseudo_kc,max,NaN,0.5804,0.6369,0.4003,1985
"pseudo_kc, noisy_or",pseudo_kc,noisy_or,NaN,0.5778,0.6383,0.3623,1985
"pseudo_kc, top2_or",pseudo_kc,top2_or,NaN,0.5804,0.6381,0.3861,1985
"pseudo_kc, mean",pseudo_kc,mean,NaN,0.5254,0.6325,0.6366,1985
"linear_pool, max",linear_pool,max,0.0303,0.6081,0.6441,0.5450,1985
"linear_pool, noisy_or",linear_pool,noisy_or,0.0181,0.6055,0.6434,0.5450,1985
"linear_pool, top2_or",linear_pool,top2_or,0.0256,0.6071,0.6439,0.5449,1985
"linear_pool, mean",linear_pool,mean,0.0000,0.6065,0.6428,0.5560,1985


## 4. Deltas against M1

Every fusion row against the baseline row's three metrics, the frozen record itself, sorted by AUC delta.

In [4]:
baseline = results.loc["baseline"]
deltas = results.drop(index="baseline").copy()
for metric in ("accuracy", "auc", "f1"):
    deltas[f"d_{metric}"] = (deltas[metric] - baseline[metric]).round(4)
deltas = deltas.sort_values("d_auc", ascending=False)
deltas[["parameter", "d_accuracy", "d_auc", "d_f1"]]

,parameter,d_accuracy,d_auc,d_f1
row,,,,
"linear_pool, max",0.0303,0.0016,0.0013,-0.0110
"linear_pool, top2_or",0.0256,0.0006,0.0011,-0.0111
"product, mean",NaN,-0.0196,0.0007,-0.2205
"linear_pool, noisy_or",0.0181,-0.0010,0.0006,-0.0110
"frozen_gate, max",0.0072,-0.0025,0.0003,-0.0098
"frozen_gate, top2_or",0.0048,-0.0005,0.0002,-0.0049
"frozen_gate, mean",0.0180,-0.0015,0.0002,-0.0071
"frozen_gate, noisy_or",0.0011,0.0016,0.0001,0.0009
"linear_pool, mean",0.0000,0.0000,0.0000,0.0000


## 5. Interpretation ledger

- The baseline row must read M1's paper-aligned trio exactly, it is the same walk on the same parameters, any difference is a wiring fault, stop and diagnose.
- Section 2's residual correlation is the budget, a warning level for linear-ish fusion rather than a bound on nonlinear methods. If it sits near the adopted 0.06, the fitted scalars are expected small, the zero-parameter rows are expected at or below baseline by dilution, product carries a 0.597-grade ranker at full weight, and any delta beyond a few thousandths would be the surprise worth chasing.
- The frozen_gate beta is the number to record regardless of metrics, fitted with no joint EM to absorb it, its distance from the early-fusion 0.178 measures what the refit equilibrium was hiding, and its distance from 1 prices the present-implies-incorrect reading at the overlay grain.
- The linear_pool w is the honest weight the data assigns the channel in the best convex blend, w near zero is the fusion-axis verdict in one number.
- Fitted scalars estimate on train log-likelihood over all scored training turns, first turns included by ruling, M1's own EM consumed them so every pipeline stage fits on the same population, the first-turn exclusion stays an evaluation convention only. The alternative, fitting on the excluded population, moves the linear-max weight 0.030 to 0.040 with no conclusion changed. Zero sits inside the feasible set so a method can select the baseline exactly, a fitted zero is the optimizer rejecting the channel outright. The training predictions are in-sample, an optimism that makes any null conservative, out-of-fold stacking is the stricter future-work design.

## 6. Why late fusion did not improve prediction

The review's audit located the failure precisely, not in alignment, which it verified end to end, but in the overlay identity itself and in what the chains' forecast actually is.

**The identity equates two different probabilities.** Same-turn annotations are strongly associated with correctness, any-family-P turns run 87.5 per cent incorrect against 15.7 for clean turns, but the overlay cannot use the current turn without leakage, it uses the chains' causal forecast of it, and that forecast correlates with current error at only 0.11 and with M1's residual at 0.05 to 0.06. The annotations carry the signal, the forecast of them does not carry enough of it.

**The forecast is saturated.** Under the chain of record the fitted dwell times run 11 to 89 turns against dialogues far shorter, so an activated chain stays near active for the rest of the dialogue, the pooled max q has median 0.885 and mean 0.748 on scored test turns, and max pooling behaves as a misconception-occurred-earlier flag rather than a turn-level error predictor. Noisy-OR is worse for the same reason, median 0.918, nearly always near one.

**Within dialogues the signal reverses.** The raw residual correlation is 0.065, and after removing each dialogue's mean it is minus 0.15, high-risk dialogues are harder, which BKT already knows, and inside a dialogue a higher chain state does not mark the erring turns. The per-family residual correlations show there was little for pooling to find, comprehension 0.038 down to steps at minus 0.006.

**The fusion arithmetic pushes the wrong way.** The baseline already predicts too few correct turns, 41.3 per cent predicted against 47.4 actual, and every overlay formula can only push probabilities down, the product row at the median q multiplies predictions by roughly 0.115 and its f1 collapses to 0.15 under max and 0.08 under noisy-OR. The fitted methods say the same thing as rejection signals, the linear weight 0.03 to 0.04 across fitting conventions, the frozen-gate beta 0.007, the mean-pooled weight at zero.

**The best row is not evidence.** Linear pool with max gains under two thousandths of AUC over M1 under either fitting convention with f1 down, and a dialogue-level bootstrap on the pre-fix row spans zero, minus 0.0006 to plus 0.0033, under test-set selection across sixteen rows. Two implementation faults found in review are fixed in the script, the fitting population now matches the scored population and zero sits inside the parameter bounds so a method can select the baseline exactly, and one is recorded as a limitation rather than fixed, the training predictions are in-sample, an optimism under which the null is conservative, out-of-fold stacking being the stricter design.

**Conclusion.** The late-fusion assumption fails, not the plumbing. The chains' causal forecast is saturated, predominantly between-dialogue, redundant with the correctness history M1 already conditions on, and mis-read as a calibrated error probability by the overlay identity. The fusion axis closes with the same verdict at both ends, and the same-turn 87.5 per cent is the ceiling the causal design correctly declines to fake.

## 7. Notes

- Nothing in this notebook fits BKT parameters or chain parameters, the overlay is a pure post-hoc layer, rerunning it is minutes.
- The threshold veto variant, q above a cutoff forcing incorrect, was considered and skipped, a discretized product row with predictable tie-collapse costs, on record in the design discussion.
- This grid completes the fusion axis, early fusion in notebooks 05 to 08, late fusion here, and its outcome goes to the report's integration verdict either way.